# This notebook will serve as a way to clean the remaining txt that will be used for char generation

In [75]:
import re

In [76]:
with open("corpus_RNN_Médine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

First we will remove the indication of featuring of the corpus

In [77]:
corpus = re.sub(r">(True|False)\n", ">\n", corpus)

Then we will reduce every space that aren't necessary

In [78]:
corpus = corpus.strip()

In [79]:
corpus_splitted = re.split(r"\n", corpus, flags=re.DOTALL)
new_list = []
for i in range(len(corpus_splitted)) :
    stripped = corpus_splitted[i].strip()
    if stripped != "" :
        if stripped == "<END>" :
            new_list.extend([stripped+"\n\n"])
        else :
            new_list.extend([stripped+"\n"])
corpus = "".join(new_list)

As the task will be char generation, first I will let indications such as INTRO, etc... in order to maybe have a RNN that can maybe identify what type of text came after Refrain (more repetitive) or after Couplet

In [81]:
corpus[:200]

"<BEGINNING>\n<INTRO>\nC'est nous l'Grand Paris\nC'est nous l'Grand Paris\n<END_SECTION>\n<COUPLET>\nLa banlieue, c'est des pâquerettes sur un tas d'fumier\nLes voix du ghetto sont auto-tunées\n<END_SECTION>\n<"

In [82]:
print(len(set(corpus)),sorted(set(corpus)))

130 ['\n', ' ', '!', '#', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\x80', '\x99', '«', '·', '»', 'À', 'Ç', 'È', 'É', 'Ê', 'Ô', 'Ö', 'à', 'á', 'â', 'ã', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'î', 'ï', 'ñ', 'ô', 'ö', 'ù', 'û', 'ü', 'Ā', 'ā', 'ğ', 'ō', 'Œ', 'œ', 'ū', 'ʿ', 'е', '–', '‘', '’', '“', '”', '…', '\ufeff']


## Corpus cleaning

Some parts are empty of lyrics because they were used just for training on the structure

In [83]:
print(re.findall(r"<.*>\n<END_SECTION>", corpus)[:5])

corpus = re.sub(r"<.*>\n<END_SECTION>\n","",corpus)

['<PONT>\n<END_SECTION>', '<COUPLET>\n<END_SECTION>', '<PONT>\n<END_SECTION>', '<COUPLET>\n<END_SECTION>', '<PONT>\n<END_SECTION>']


End_Section can be replaced by a specific character that isn't in the corpus such as §

In [84]:
corpus = re.sub(r"\n<END_SECTION>\n","\n§\n", corpus)

Indication of beginning and ending of a song isn't required, the model learn only the lyrics structure inside a part not a song structure

In [85]:
corpus = re.sub(r"<BEGINNING>\n","", corpus)
corpus = re.sub(r"<END>\n","", corpus)

In [86]:
print(re.findall(r"\n\w+\n", corpus)[:10],len(re.findall(r"\n\w+\n", corpus))) 

['\nCheck\n', '\nAlgérie\n', '\nAlgérie\n', '\nAlgérie\n', '\nVerse\n', '\nLes\n', '\nGIA\n', '\nLH\n', '\nPégase\n', '\nTomahawk\n'] 113


In [87]:
corpus = re.sub(r"\n([a-z]\w+)\n", r"\1\n", corpus) #If lowercase, we assume that the sentence wasn't finished
corpus = re.sub(r"\n([A-Z]\w+)\n", r"\n\1", corpus) #If uppercase, we assume we are at the beginning of the sentence

In [88]:
print(re.findall(r"\n\w+\n", corpus)[:10],len(re.findall(r"\n\w+\n", corpus))) 

['\nLibérezMumiaet\n', '\nAboubakretYahia\n', '\nEnarceetKoto\n', '\nChicanosetRemos\n', '\nÉcoutes\n', '\nStopMédine\n', '\nMédineBrav\n', '\n2011\n', '\n1427\n', '\n1427\n'] 12


### Characters : <>

In [89]:
set(re.findall(r"\<.*\>", corpus)) #No need to modify this

{'<COUPLET>', '<INTRO>', '<OUTRO>', '<PONT>', '<REFRAIN>'}

### Characters : ()

In [90]:
print(re.findall(r"\(.*?\)", corpus, flags=re.DOTALL)[:5])
corpus = re.sub(r"\(.*?\)", "", corpus, flags=re.DOTALL)

["(\nCaucri'\n)", "(\nBois d'Blé'\n)", "(A visage découvert, dans la rue, dans l'rap, on r'met les couverts\nRefaire le monde à notre sauce, soldat la guerre est ouverte)", '(Baccardi, Médine, on plante le décor\nPremière classe, DIN Records, la fin du monde, nouvel épisode)', "(A visage découvert, dans la rue, dans l'rap, on r'met les couverts\nRefaire le monde à notre sauce, soldat la guerre est ouverte)"]


### Character : *

In [92]:
print(re.findall(r"\*.*", corpus))
corpus = re.sub(r"\*", "", corpus, flags=re.DOTALL)

['*Spartiate, quel est votre métier? Aouh, aouh, aouh!*', '*Détonation*', '*Femme pleurant au téléphone*', "*Femme expliquant la situation avant qu'une explosion retentisse*"]


### Characters : « »

In [93]:
print(len(re.findall("«.*?»", corpus)),re.findall("«.*?»", corpus)[:5])
#This lyrics are just citation such as TV journals having a reportage on Medine
corpus = re.sub("«.*?»", "", corpus)

50 ['« Nietzsche est mort »', "« Non mais vous dîtes il parle dans l'absolu, ça n'est pas vrai...»", "«...Ces vociférations, je l'avoue, m'inquiètent...»", '«...Son rap à mon avis joue avec le feu...»', "«J'ai pas tout saisi de ce déchaînement verbal, il faut élargir la question. Le rap est, à quelques exceptions près, il y en a je le reconnais, un véritable dégueulis verbal d'une violence extrême»"]


### Characters : “ ”

In [94]:
print(re.findall(".*”", corpus))

corpus = re.sub("”", '"', corpus)
corpus = re.sub("“", '"', corpus)

['Comme dans “La vie est belle”', "J'te souhaite la bienvenue au Havre , autrefois appelé “Franciscopolis”"]


### Character : …

In [95]:
print(re.findall(".*….*", corpus))

corpus = re.sub("…", "...", corpus)

['La terre fleurit dès que je l’ai foulée…', 'La terre fleurit dès que je l’ai foulée…', 'La terre fleurit dès que je l’ai foulée…', 'La terre fleurit dès que je l’ai foulée…', 'Réponse: Trois petits points de suspension…', 'Et vas-y, euh, rase-toi la barbe…', 'Comme les responsables…']


### Characters : ‘ ’

In [96]:
print(re.findall(".*‘.*",corpus))

corpus = corpus.replace("‘", "'")

['C‘est pour les Russel Crowe qui recèlent trop', 'Où l‘on s‘achète un pavillon avec une sonnerie polyphonique', 'Mais l‘histoire nous rattrapa, car troqués en pesetas', 'Qui ser-‘cra, même le pire des scards-la sant-cri', "C‘est dur de voir l'adolescence en manque de souffle"]


In [97]:
print(len(re.findall(".*’.*",corpus)),re.findall(".*’.*",corpus)[:5])

corpus = re.sub("’", "'", corpus)

1056 ["T'as la vision d'un journaliste alors je t’appelle le presbyte", 'T’as pas de grands frères, de grands hommes auxquels t’identifier', 'T’as pas de grandes guerres, de grandes causes auxquelles te sacrifier', 'Pas de gangsters, de grands guns pour s’authentifier', 'Sous nos grands airs de grandes gueules on vient pour s’unifier']


### Characters : е (different than normal e)

In [99]:
print(re.findall(".*е.*", corpus)[:5])

corpus = re.sub("е", "e", corpus)

["J'ai dû m'éloigner d'leurs mosquées pour mе rapprocher de Dieu", "Ils s'battеnt pour les pouvoirs comme sur l'rrain-te pour un 10 eu'", "Mais faudra vraiment être sûr de toi parce que c'que tu diras sur scène, le public s'еn emparera et s'y connеctera. Les gens s'approprient en tes mots. Ils s'en empareront. Et à partir de ce moment-là, ils s'envoleront pour toujours", 'Ce sera jamais nous contre eux, les mauvaises hеrbes sont des fleurs', "On n'a pas bеsoin d'thérapeutes, un comedy club dans le cœur"]


### Character : \#

In [100]:
print(re.findall(".*#.*", corpus))

corpus = re.sub("#", "",  corpus)

['RT, #dièse, NGRTD', 'Frappe les PAO, gagne par KO! #MannyPacquiao', "La France j'y suis, j'y reste. Anti-colonial #JeanJaurès", "On n'innocente pas le petit Ahmed avec un #NotInMyName", "Comme Kanye j'inclus les handicapés #Oups", '#FreePalestine #AuFreestyleDeSky', "Depuis que la zik s'écoute que sur les sites bientôt ton boss s'appelle Fif Tobossi #Oups", "Le seul slogan qu'on voit c'est hashtag #FreeLacrim", "OrelsanJe porte un toast à la mort de l'industrieSortez les 8'6, on vient fêter la fin du disqueÉcouter la radio c'est devenu un suppliceSauf que j'aime pas non plus les putains de puristesMusique rétro-futuristeLa bande originale des aventures d'UlysseJ'habiterais dans les abysses j'aurais pas plus de pressionTout ce que je veux : foutre le feu dans ma ville #NéronDonner mon corps à la science inclut la dissectionDes jours entiers je récite mes leçons rourmenté dans une pluie de questionsRien qu'en un an ça m'a soûlé, je voulais tout plaquer, quitter le sonJ'ai presque aband

### Character : "

In [101]:
print(re.findall('.*".*', corpus))

corpus = re.sub('"', "", corpus)

['Comme dans "La vie est belle" d\'Roberto Benigni', 'J\'te souhaite la bienvenue au Havre , autrefois appelé "Franciscopolis"']


### \(

In [102]:
corpus = re.sub("\(", "", corpus)

## »|«

In [103]:
print(re.findall(".*»|«.*", corpus))

corpus = re.sub("»|«", "", corpus)

['« Pourquoi détruire les maisons des Palestiniens', '« La CIA nous a bien renseigné', "Dans les caves d'une école c'est là qu'ils sont cachés »", '« Mes parents sont insensibles et cruels', "C'est décidé! Il faut que je rentre chez moi »", "« Wallah Medine bsahtek ouah j'ai écouté wallah t'as raison", "Wallah Faut qu'ils payent wallah »", '« Médine Médine', 'Médine Médine »', '« Médine Médine', 'Médine Médine »', '« Médine Médine', 'Médine Médine »', '« Double discours', 'Petit journaliste en stage »', '« Tu te rapproches de la ratonnade bicot', "Voudrais-tu t'attirer les foudres? »", '« Te voici dans la maison du Seigneur', 'Tu es ici chez toi pour le temps désirant »']


## \x80

In [104]:
print(re.findall(".*\x80.*", corpus))

corpus = re.sub("\x80", "", corpus)

["Nous n'voulions pas non plus d\x80'une Algérie Française", "Lorsque ma part algérienne s'\x80\x99exprime dans le micro d'\x80\x99la vie"]


## \x99

In [105]:
print(re.findall(".*\x99.*", corpus))

corpus = re.sub("\x99", "", corpus)

["Lorsque ma part algérienne s'\x99exprime dans le micro d'\x99la vie"]


### '

In [106]:
re.findall("'\s", corpus)[:10]

["' ", "'\n", "'\n", "'\n", "' ", "'\n", "' ", "' ", "' ", "'\n"]

In [107]:
def remove_special(match):
    return match.group(0).replace("'", "")

corpus = re.sub(r"'\s|\s+'", remove_special, corpus, flags=re.DOTALL)

In [108]:
re.findall(".*Ā.*", corpus)

['Ou finira comme les peuples des ʿĀd et des Thamūd']

## \ufeff

In [109]:
print(re.findall(".*\ufeff.*", corpus))

corpus = re.sub("\ufeff", "", corpus)

['Proof et maître Henni\ufeff-Mansour']


### ʿ

In [110]:
print(re.findall(".*ʿ.*", corpus))

corpus = re.sub("ʿ", "", corpus)

['Ou finira comme les peuples des ʿĀd et des Thamūd']


### ·

In [111]:
print(re.findall(".*·.*", corpus))

corpus = re.sub("·", "", corpus)

['Propriété intellectuelle TA3 les convaincu·es', 'Entre nous, entre convaincu·es, pas de danse du ventre']


In [112]:
re.findall(r"\w+ğ\w+", corpus)

['Erdoğan']

In [113]:
print(len(set(corpus)),sorted(set(corpus)))

114 ['\n', ' ', '!', '%', '&', "'", ')', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '§', 'À', 'Ç', 'È', 'É', 'Ê', 'Ô', 'Ö', 'à', 'á', 'â', 'ã', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'í', 'î', 'ï', 'ñ', 'ô', 'ö', 'ù', 'û', 'ü', 'Ā', 'ā', 'ğ', 'ō', 'Œ', 'œ', 'ū', '–']


Ok the txt seems good for now

In [38]:
with open(f"clean_corpus_Medine.txt", "w", encoding="utf-8") as f :
    f.write(corpus)